# HW4 — Binary Key-Value Store: Requirements & Tests

## Requirements Checklist

### Server (`key-value-server-binary`)
| # | Requirement | Status |
|---|-------------|--------|
| 1 | Replace `std::vector` store with `std::unordered_map<string,string>` | ✅ |
| 2 | Protect shared store with `std::mutex` (thread-safe) | ✅ |
| 3 | Handle multiple clients concurrently via `std::thread` | ✅ |
| 4 | **PUT** opcode (0x01): read key+value, insert/overwrite, respond OK | ✅ |
| 5 | **GET** opcode (0x02): read key, respond VALUE or NOT_FOUND | ✅ |
| 6 | **DELETE** opcode (0x03): read key, erase, respond OK or NOT_FOUND | ✅ |
| 7 | **COUNT** opcode (0x04): respond with integer count of pairs | ✅ |
| 8 | **EXISTS** opcode (0x05): respond OK if key present, NOT_FOUND otherwise | ✅ |
| 9 | **KEYS** opcode (0x06): respond with count + all keys as length-prefixed strings | ✅ |
| 10 | Respond ERROR (0x7F) for unknown/malformed requests | ✅ |
| 11 | Wire format: 4-byte big-endian length prefix + payload (opcode byte + args) | ✅ |
| 12 | Integer args/results encoded as 4-byte big-endian (htonl/ntohl) | ✅ |
| 13 | String args/results encoded as int32 length + ASCII bytes | ✅ |

### Client stub (`key-value-client-binary`)
| # | Requirement | Status |
|---|-------------|--------|
| 14 | `RemoteListStub` — opcode-agnostic transport (sendRequest) | ✅ |
| 15 | `RemoteKeyValueStore` — semantic layer wrapping the stub | ✅ |
| 16 | `put(key, value) -> bool` | ✅ |
| 17 | `get(key) -> optional<string>` (nullopt if NOT_FOUND) | ✅ |
| 18 | `remove(key) -> bool` | ✅ |
| 19 | `count() -> optional<size_t>` | ✅ |
| 20 | `exists(key) -> bool` | ✅ |
| 21 | `keys() -> optional<vector<string>>` | ✅ |

### Client `main.cpp`
| # | Requirement | Status |
|---|-------------|--------|
| 22 | **Option 1**: read "key value" lines, PUT each, then print all keys | ✅ |
| 23 | **Option 2**: GET all values via keys(), sort alphabetically, print | ✅ |
| 24 | Connects to `127.0.0.1:9090` | ✅ |


## Setup

Build the server and client first:
```bash
# From repo root
cmake -S RPC/key-value-server-binary -B RPC/key-value-server-binary/build
cmake --build RPC/key-value-server-binary/build

cmake -S RPC/key-value-client-binary -B RPC/key-value-client-binary/build
cmake --build RPC/key-value-client-binary/build
```

Then start the server in a separate terminal:
```bash
./RPC/key-value-server-binary/build/key_value_server_binary
```

The cells below connect directly via raw sockets — **no server restart needed between cells**.

In [ ]:
import socket
import struct

HOST = '127.0.0.1'
PORT = 9090

# Opcodes (must match server)
OP_PUT    = 0x01
OP_GET    = 0x02
OP_DELETE = 0x03
OP_COUNT  = 0x04
OP_EXISTS = 0x05
OP_KEYS   = 0x06

RESP_OK       = 0x40
RESP_VALUE    = 0x41
RESP_COUNT    = 0x42
RESP_KEYS     = 0x43
RESP_NOTFOUND = 0x44
RESP_ERROR    = 0x7F

def encode_int32(n: int) -> bytes:
    return struct.pack('>i', n)

def encode_string(s: str) -> bytes:
    b = s.encode('ascii')
    return encode_int32(len(b)) + b

def frame(opcode: int, payload: bytes) -> bytes:
    body = bytes([opcode]) + payload
    return struct.pack('>I', len(body)) + body

def send_recv(opcode: int, payload: bytes = b'') -> tuple[int, bytes]:
    with socket.create_connection((HOST, PORT)) as sock:
        sock.sendall(frame(opcode, payload))
        length_bytes = sock.recv(4)
        if len(length_bytes) < 4:
            raise RuntimeError('Short read on length')
        length = struct.unpack('>I', length_bytes)[0]
        data = b''
        while len(data) < length:
            chunk = sock.recv(length - len(data))
            if not chunk:
                break
            data += chunk
    resp_opcode = data[0]
    resp_payload = data[1:]
    return resp_opcode, resp_payload

def decode_string(data: bytes, offset: int) -> tuple[str, int]:
    length = struct.unpack_from('>i', data, offset)[0]
    offset += 4
    return data[offset:offset+length].decode('ascii'), offset + length

def decode_int32(data: bytes, offset: int) -> tuple[int, int]:
    return struct.unpack_from('>i', data, offset)[0], offset + 4

passed = 0
failed = 0

def check(name, condition):
    global passed, failed
    if condition:
        print(f'  PASS  {name}')
        passed += 1
    else:
        print(f'  FAIL  {name}')
        failed += 1

print('Protocol helpers ready. Make sure the server is running on port 9090.')

In [ ]:
# ── PUT ──────────────────────────────────────────────────────────────────────
print('=== PUT ===')

op, _ = send_recv(OP_PUT, encode_string('color') + encode_string('blue'))
check('PUT color=blue -> OK', op == RESP_OK)

op, _ = send_recv(OP_PUT, encode_string('fruit') + encode_string('apple'))
check('PUT fruit=apple -> OK', op == RESP_OK)

op, _ = send_recv(OP_PUT, encode_string('color') + encode_string('red'))
check('PUT color=red (overwrite) -> OK', op == RESP_OK)

In [ ]:
# ── GET ──────────────────────────────────────────────────────────────────────
print('=== GET ===')

op, payload = send_recv(OP_GET, encode_string('color'))
check('GET color -> VALUE opcode', op == RESP_VALUE)
val, _ = decode_string(payload, 0)
check('GET color -> "red" (overwrite took effect)', val == 'red')

op, payload = send_recv(OP_GET, encode_string('fruit'))
check('GET fruit -> VALUE opcode', op == RESP_VALUE)
val, _ = decode_string(payload, 0)
check('GET fruit -> "apple"', val == 'apple')

op, _ = send_recv(OP_GET, encode_string('missing'))
check('GET missing key -> NOT_FOUND', op == RESP_NOTFOUND)

In [ ]:
# ── COUNT ─────────────────────────────────────────────────────────────────────
print('=== COUNT ===')

op, payload = send_recv(OP_COUNT)
check('COUNT -> COUNT opcode', op == RESP_COUNT)
n, _ = decode_int32(payload, 0)
check(f'COUNT -> 2 pairs (got {n})', n == 2)

In [ ]:
# ── EXISTS ────────────────────────────────────────────────────────────────────
print('=== EXISTS ===')

op, _ = send_recv(OP_EXISTS, encode_string('color'))
check('EXISTS color -> OK', op == RESP_OK)

op, _ = send_recv(OP_EXISTS, encode_string('ghost'))
check('EXISTS ghost -> NOT_FOUND', op == RESP_NOTFOUND)

In [ ]:
# ── KEYS ──────────────────────────────────────────────────────────────────────
print('=== KEYS ===')

op, payload = send_recv(OP_KEYS)
check('KEYS -> KEYS opcode', op == RESP_KEYS)
count, offset = decode_int32(payload, 0)
keys = []
for _ in range(count):
    k, offset = decode_string(payload, offset)
    keys.append(k)
check(f'KEYS -> 2 keys (got {count})', count == 2)
check(f'KEYS contains "color" and "fruit": {sorted(keys)}', sorted(keys) == ['color', 'fruit'])

In [ ]:
# ── DELETE ────────────────────────────────────────────────────────────────────
print('=== DELETE ===')

op, _ = send_recv(OP_DELETE, encode_string('fruit'))
check('DELETE fruit -> OK', op == RESP_OK)

op, _ = send_recv(OP_GET, encode_string('fruit'))
check('GET fruit after DELETE -> NOT_FOUND', op == RESP_NOTFOUND)

op, _ = send_recv(OP_DELETE, encode_string('fruit'))
check('DELETE fruit again -> NOT_FOUND', op == RESP_NOTFOUND)

op, payload = send_recv(OP_COUNT)
n, _ = decode_int32(payload, 0)
check(f'COUNT after DELETE -> 1 (got {n})', n == 1)

In [ ]:
# ── ERROR / malformed ─────────────────────────────────────────────────────────
print('=== ERROR handling ===')

op, _ = send_recv(0xFF)  # unknown opcode
check('Unknown opcode 0xFF -> ERROR', op == RESP_ERROR)

op, _ = send_recv(OP_PUT, encode_string('only_key'))  # missing value
check('PUT with missing value -> ERROR', op == RESP_ERROR)

In [ ]:
# ── Two-client simulation (Option 1 then Option 2 behaviour) ─────────────────
print('=== Two-client simulation ===')

# Client 1: PUT several pairs
pairs = [('animal', 'zebra'), ('city', 'miami'), ('season', 'autumn')]
for k, v in pairs:
    op, _ = send_recv(OP_PUT, encode_string(k) + encode_string(v))
    check(f'Client1 PUT {k}={v}', op == RESP_OK)

# Client 1: list keys
op, payload = send_recv(OP_KEYS)
count, offset = decode_int32(payload, 0)
all_keys = []
for _ in range(count):
    k, offset = decode_string(payload, offset)
    all_keys.append(k)
check('Client1 KEYS returns 4 keys (color + 3 new)', count == 4)

# Client 2: GET all values, sort, print
op2, payload2 = send_recv(OP_KEYS)
count2, offset2 = decode_int32(payload2, 0)
values = []
for _ in range(count2):
    k, offset2 = decode_string(payload2, offset2)
    op_v, pv = send_recv(OP_GET, encode_string(k))
    if op_v == RESP_VALUE:
        v, _ = decode_string(pv, 0)
        values.append(v)
values.sort()
print(f'  Client2 values (sorted): {values}')
check('Client2 sorted values match expected', values == sorted(['red', 'zebra', 'miami', 'autumn']))

In [ ]:
# ── Summary ───────────────────────────────────────────────────────────────────
total = passed + failed
print(f'\n{"="*40}')
print(f'Results: {passed}/{total} passed')
if failed == 0:
    print('All tests passed!')
else:
    print(f'{failed} test(s) FAILED — check output above.')
print('=' * 40)